In [7]:
pip install transformers

In [8]:
pip install "transformers[torch]"

In [9]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration


In [10]:

train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")


In [11]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [12]:

train_data.sample(10)

,id,dialogue,summary
7311,13728394,"Joe: Amy, why is New York named New York?\r\nA...",New York used to be called New Amsterdam but t...
7631,13818135,Madison: What do you think about the professor...,Ethan has suggested to Alexander that Madison ...
9317,13682113,Sandy: What did u get?\r\nTrish: 3 :(\r\nSandy...,Sandy and Trish both got 3.
368,13810048,Charity: hey\r\nJoyce: hey\r\nCharity: is the ...,Charity wants to know if HOD is in but he has ...
5695,13727779,Abdi: salam alyekum\r\nRashid: alyekum salam\r...,"Rashid's son, Yusuf, went to high school. Huss..."
691,13865106,Cheryl: Have you seen Jenny recently?\nCristin...,Jenny is in town but she's busy with the baby....
3676,13821858,"Marie: girls, I'm in Bologna\r\nSarah: is it ...","Marie is in Bologna, and she finds the local m..."
5901,13828620,Laura: do you have the results of your history...,Zach did poorly on his history exam and is wor...
1338,13680370,Ron: pizza for dinner?\r\nFelicia: ok\r\nRon: ...,Ron offers to order pizza for dinner. He wants...
5749,13728115,Erik: Hey\r\nErik: Hello \r\nAlex: hey hey wha...,Alex will probably set notifications on his ph...


In [13]:
train_data.shape

(14732, 3)

In [14]:
val_data.shape

(818, 3)

In [15]:
# random sampling --

train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)


# Preprocessing --

In [16]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.strip().lower()
    return text



In [17]:

train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summmary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = train_data["dialogue"].apply(clean_data)
val_data["summmary"] = train_data["summary"].apply(clean_data)


In [18]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

# Tokenization--


In [19]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [20]:
# raw data => tokenized inp for => fine tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)

    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"]
    return inputs


In [21]:

train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()


In [22]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [23]:
# input ids - dialogue => token ids

#1 => EOS, 0=> padding

# attention mask
# labels - target => summary token

len(train_dataset[0]["input_ids"])


512

In [24]:
type(train_dataset)
type(val_dataset)

list

# Working With Our Model -


In [25]:
# training is done !

# NLP => generation task-

model = T5ForConditionalGeneration.from_pretrained("t5-small")


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [26]:
# fine - tune =>

import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else :
    device = torch.device("cpu")

print("device: ",device)
model.to(device)


device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [30]:

# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500
)

In [31]:

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [32]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.346593,0.848747
2,0.472347,0.849011
3,0.443960,0.855433
4,0.429583,0.859523
5,0.420702,0.864929
6,0.416180,0.863521


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.5882275746663411, metrics={'train_runtime': 1338.4381, 'train_samples_per_second': 17.931, 'train_steps_per_second': 2.241, 'total_flos': 3248203235328000.0, 'train_loss': 0.5882275746663411, 'epoch': 6.0})

In [33]:
# model load => fine-tune => save the model

In [34]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [35]:
# Test the core logic for summarization =>

In [36]:
def summarize_dialogue(dialogue):
  #clean
  dialogue = clean_data(dialogue)

  #tokenize
  inputs = tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt"
  ).to(device)

  #generate summary =>
  model.to(device)
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length=150,
      num_beams = 4, #4 diff- oup compare honge
      early_stopping=True
  )

  #token ids convert to summary => decoding
  summary = tokenizer.decode(targets[0], skip_special_tokens=True)  #EOS, SEP Tab
  return summary


In [37]:

test_dialogue = """
Rahul: Hey Priya, did you attend the project meeting this morning?

Priya: Yes, I did. The team discussed the progress of the text summarization project.

Rahul: That's good. Did the manager mention anything about the deadline?

Priya: Yes. The final project needs to be completed by Friday. We also need to prepare a short presentation for the demonstration.

Rahul: Are we finished with the model training?

Priya: Not yet. The model is currently being trained, and the team plans to use a GPU to reduce the training time.

Rahul: What about the application?

Priya: The basic application is ready. Users can enter a conversation or paragraph, and the application generates a short summary.

Rahul: Great. Do we need to improve anything?

Priya: We should improve the summary quality, test the application with longer conversations, and fix any errors before the final demonstration.

Rahul: Okay. I'll work on testing the application today.

Priya: I'll work on improving the model and preparing the presentation.

Rahul: Perfect. Let's meet tomorrow and review our progress.

Priya: Sounds good. See you tomorrow.

"""


summary = summarize_dialogue(test_dialogue)
print("Summary :",summary)


Summary : Rahul attends the project meeting this morning. The team discussed the progress of the text summarization project. The final project needs to be completed by Friday. The model is currently being trained, and the team plans to use a gpu to reduce the training time.
